# Bruit multi-sources Suisse + ORHANE (classes 1 à 7) — Grand Genève

*Version multibruit côté suisse : Claude, 26.11.2026. La version précédente ne prenait côté suisse que le bruit routier de jour.*

Ce notebook construit **une couche de bruit unique pour tout le Grand Genève**, en classes ORHANE de 1 à 7 :

- **Côté France :** on reprend l'indicateur multibruit ORHANE tel quel (route, fer et aérien déjà combinés).
- **Côté Suisse :** on reconstruit un **multibruit équivalent** avec la même méthode qu'ORHANE, à partir des données fédérales (route, fer et aérien, de jour et de nuit).

Les deux sont ensuite réunis sur une grille commune de 10 m en EPSG:2056. Là où les deux couches ont une valeur (à la frontière), on garde le **max**.

## 1. Données

| Source | Couche | Indicateur | Producteur |
|---|---|---|---|
| Route CH | `laerm-strassenlaerm_tag_2056.tif`, `laerm-strassenlaerm_nacht_2056.tif` | Lr jour (6h–22h) et Lr nuit (22h–6h), dB(A), raster 10 m | OFEV, sonBASE (état 2021) |
| Fer CH | `laerm-bahnlaerm_tag_2056.tif`, `laerm-bahnlaerm_nacht_2056.tif` | Lr jour et Lr nuit, dB(A), raster 10 m | OFEV, sonBASE (état 2021) |
| Aérien CH | `laermbelastungskataster-zivilflugplaetze_2056.gpkg` | Courbes d'égal Lr (lignes fermées, champ `LEVEL_DB`) : jour 6h–22h (`OverallTrafficDay_Lr`, sinon `LightAircraft_Lr`) et heures de nuit 22h–23h, 23h–24h et 5h–6h (`FirstNightHour_Lr`, `SecondNightHour_Lr`, `LastNightHour_Lr`) | OFAC, cadastres de bruit des aérodromes civils |
| France | `orhane_2023_multibruit/sous_indice_multibruit_aura.tif` | Classes multibruit 1 à 7 (-99 = hors région) | ATMO Auvergne-Rhône-Alpes / Acoucité / Cerema |

## 2. Méthode côté suisse (équations)

**(1) Lden approché pour chaque source, à partir des Lr suisses.** La législation suisse (OPB, annexes 3 à 5) définit un jour de 6h à 22h (16 h) et une nuit de 22h à 6h (8 h). Il n'y a pas de période de soirée.

$$L_{den} \approx 10\,\log_{10}\!\left[\frac{16\cdot10^{L_{r,j}/10} + 8\cdot10^{(L_{r,n}+10)/10}}{24}\right]$$

Une période sans donnée (NaN) compte pour une énergie nulle : pas de train la nuit, pas d'avion hors des courbes, etc.

**(2) Nuit aérienne.** Le cadastre de l'OFAC donne un Lr pour **chaque heure** de nuit avec vols (22h–23h, 23h–24h, 5h–6h). On le ramène à 8 h, les heures sans courbe comptant pour 0 :

$$L_{r,n,air} = 10\,\log_{10}\!\left[\frac{1}{8}\sum_{h} 10^{L_{r,h}/10}\right]$$

Un pixel situé dans la courbe de niveau *L* (et dans aucune courbe plus élevée) reçoit la valeur *L* : c'est le niveau plancher de la bande. Les aérodromes sont combinés par somme énergétique.

**(3) Courbes dose-effet de l'OMS 2018 pour la forte gêne** (Directive (UE) 2020/367, annexe III ; arrêté français du 23.12.2021, formules 4 à 6), avec *L* = Lden en dB(A) et %HA = pourcentage de personnes fortement gênées :

$$\%HA_{route} = 78{,}9270 - 3{,}1162\,L + 0{,}0342\,L^2$$
$$\%HA_{fer} = 38{,}1596 - 2{,}05538\,L + 0{,}0285\,L^2$$
$$\%HA_{air} = -50{,}9693 + 1{,}0168\,L + 0{,}0072\,L^2$$

**(4) Niveau « équivalent gêne routière »,** comme ORHANE : on cherche le Lden routier qui donne la même %HA que la source, en inversant la courbe routière (racine croissante) :

$$L_{eq} = \frac{3{,}1162 + \sqrt{3{,}1162^2 - 4\cdot0{,}0342\cdot\left(78{,}9270 - \%HA_{source}(L)\right)}}{2\cdot0{,}0342}$$

La courbe routière n'est inversible qu'au-dessus de son minimum, situé à 45,6 dB pour 7,94 % de gêne. En dessous, ou si %HA de la source est inférieur à 7,94 %, on garde $L_{eq} = L$. On impose aussi $L_{eq} \ge L$ (`EQUIV_OFFSET_MIN_DB = 0`) : à bas niveau, la courbe ferroviaire passe sous la courbe routière, et ORHANE ne rapporte que des écarts de 0 à +3 dB pour le fer et de 0 à +13 dB pour l'aérien.

**(5) Combinaison par somme énergétique :**

$$L_{multi} = 10\,\log_{10}\!\left(10^{L_{route}/10} + 10^{L_{fer,eq}/10} + 10^{L_{air,eq}/10}\right)$$

**(6) Classes ORHANE,** avec les bornes basses incluses :

| Classe | Zone | Lden |
|---|---|---|
| 1 | Préservée ou absence de donnée | < 50 |
| 2 | Peu altérée | 50 – 55 |
| 3 | Moyennement altérée | 55 – 60 |
| 4 | Altérée | 60 – 65 |
| 5 | Dégradée | 65 – 70 |
| 6 | Très dégradée | 70 – 75 |
| 7 | Hautement dégradée | ≥ 75 |

Ensuite : `classe_norm = (classe − 1) / 6`, union avec ORHANE (max à la frontière), découpe au périmètre, puis export.

**Colonnes de contrôle** (côté CH, vides côté FR) :
- `lden_route`, `lden_fer`, `lden_air` : Lden de chaque source, équation (1) ;
- `lden_fer_eq`, `lden_air_eq` : niveaux équivalent route, équation (4) ;
- `lden_multi_ch` : équation (5).

Le même contenu est aussi exporté dans un GeoTIFF à 6 bandes, `*_lden_ch.tif`.

## 3. Limites

- **Lr n'est pas un Leq.** Le Lr suisse inclut des corrections K1 (route : jusqu'à −5 dB à faible trafic ; fer : bonification jusqu'à −15 dB). Le Lden obtenu est donc sous-estimé, surtout pour le fer.
- **Pas de soirée.** La tranche 19h–22h reste dans le jour, sans la majoration de +5 dB, soit au plus ≈ 1 dB de moins.
- **Aérien.** La valeur est plancher par bande (au pas des courbes : 1 dB à Genève, 5 dB à La Côte). Il n'y a aucune information sous la courbe la plus basse (55 dB le jour à Genève). `LightAircraft_Lr` (petits aéronefs) sert de jour quand l'aérodrome n'a pas de `OverallTrafficDay_Lr`.
- **Asymétrie de la classe 1 côté France.** Côté suisse, la classe 1 veut dire « moins de 50 dB ». Côté ORHANE, elle mélange « moins de 50 dB » et « pas de donnée » (option `FR_CLASSE_1`).

## 4. Références

- OFEV — base de données sonBASE ; cartes `ch.bafu.laerm-strassenlaerm_tag/_nacht` et `ch.bafu.laerm-bahnlaerm_tag/_nacht` (data.geo.admin.ch). « Lärmbelastung in der Schweiz. Ergebnisse des nationalen Lärmmonitorings sonBASE, Stand 2021 », OFEV 2025.
- OFAC — cadastres de bruit des aérodromes civils, `ch.bazl.laermbelastungskataster-zivilflugplaetze` (data.geo.admin.ch).
- Ordonnance sur la protection contre le bruit (OPB, RS 814.41) : annexe 3 (route), annexe 4 (chemins de fer), annexe 5 (aérodromes civils), pour la définition des Lr et des périodes.
- WHO Regional Office for Europe (2018). *Environmental Noise Guidelines for the European Region*. ISBN 978 92 890 5356 3.
- Guski R., Schreckenberg D., Schuemer R. (2017). *WHO Environmental Noise Guidelines for the European Region: A Systematic Review on Environmental Noise and Annoyance*. Int. J. Environ. Res. Public Health 14(12), 1539. Source des courbes %HA.
- Directive (UE) 2020/367 de la Commission du 4 mars 2020, modifiant l'annexe III de la directive 2002/49/CE (méthodes d'évaluation des effets nuisibles).
- Arrêté du 23 décembre 2021 modifiant l'arrêté du 4 avril 2006 relatif à l'établissement des cartes de bruit (transposition française ; formules 4 à 6).
- Cerema Centre-Est (2022). *ORHANE — note technique méthodologie de calcul multi-bruit* (Lden équivalent gêne routière, courbes OMS 2018, maille de 10 m).
- ORHANE 2023 : « Source ATMO Auvergne-Rhône-Alpes / ACOUCITE / CEREMA », licence ODbL.

## Dépendances

In [1]:
import importlib.util
import subprocess
import sys

REQUIRED = ["rasterio", "geopandas", "shapely", "pyproj", "pyarrow", "numpy", "pandas", "pyogrio"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.warp import Resampling, reproject, transform_bounds
from rasterio.windows import Window, from_bounds

print("rasterio", rasterio.__version__, "| geopandas", gpd.__version__, "| numpy", np.__version__)

rasterio 1.4.3 | geopandas 1.0.1 | numpy 2.0.0


## Paramètres

- **Chemins :** relatifs au dépôt (`Data/input/attributs/GG/bruit`).
- `CH_MULTIBRUIT` :
  - `True` : multibruit suisse (route + fer + aérien, équations 1 à 5) ;
  - `False` : ancienne méthode (route de jour seule, Lr classé directement).
- `EQUIV_GENE` : `True` convertit le fer et l'aérien en équivalent gêne routière, selon l'OMS 2018 (équation 4). Avec `False`, on fait la somme énergétique brute des Lden.
- `EQUIV_OFFSET_MIN_DB` : écart minimal imposé entre $L_{eq}$ et *L*. `0.0` donne $L_{eq} \ge L$ ; `None` désactive la borne.
- `AIR_CLOSE_TOL_M` : une courbe aérienne non fermée est fermée si l'écart entre ses deux extrémités est inférieur à cette valeur ; sinon elle est ignorée.
- `FR_CLASSE_1` : traitement des pixels ORHANE en classe 1 (« moins de 50 dB ou absence de donnée ») :
  - `"45dB"` : on les garde en **classe 1**, avec 45 dB comme valeur indicative ;
  - `"nan"` : on les considère comme **sans donnée** et on les exclut de l'export.
- `RESOLUTION_M` : taille de pixel de la grille commune (10 m, comme les sources). `None` prend la résolution native la plus grossière.
- `OVERLAP_RULE` : règle là où la Suisse et la France ont toutes deux une valeur (`"max"`, `"priorite_ch"` ou `"priorite_fr"`).
- `CH_EXTRA_NODATA` : valeurs des rasters suisses à traiter comme « pas de donnée », en plus du nodata déclaré.

In [2]:
BRUIT_DIR = Path("../../Data/input/attributs/GG/bruit")
AGREG_DIR = Path("../../Data/input/network_agreg")

# --- Suisse (OFEV sonBASE + OFAC) ---
CH_RASTERS = {                       # Lr en dB(A), raster 10 m, EPSG:2056
    "route": {"jour": BRUIT_DIR / "laerm-strassenlaerm_tag_2056.tif",
              "nuit": BRUIT_DIR / "laerm-strassenlaerm_nacht_2056.tif"},
    "fer":   {"jour": BRUIT_DIR / "laerm-bahnlaerm_tag_2056.tif",
              "nuit": BRUIT_DIR / "laerm-bahnlaerm_nacht_2056.tif"},
}
CH_AIR_GPKG = BRUIT_DIR / "laermbelastungskataster-zivilflugplaetze_2056.gpkg"
AIR_DAY_LAYERS = ["OverallTrafficDay_Lr", "LightAircraft_Lr"]       # par aérodrome : 1re couche disponible (6h-22h)
AIR_NIGHT_LAYERS = ["FirstNightHour_Lr", "SecondNightHour_Lr", "LastNightHour_Lr"]   # Lr horaires 22h-23h, 23h-24h, 5h-6h
AIR_LEVEL_FIELD = "LEVEL_DB"
AIR_CLOSE_TOL_M = 50.0

# --- France (ORHANE) ---
FR_TIF = BRUIT_DIR / "orhane_2023_multibruit" / "sous_indice_multibruit_aura.tif"   # classes 1..7, nodata -99

PERIMETER_VARIANTS = {
    "avec_lac": {"kind": "shp", "path": AGREG_DIR / "AGGLO_PERIMETRE_AVEC_LAC-SHP" / "AGGLO_PERIMETRE_AVEC_LAC.shp"},
    "sans_lac": {"kind": "parquet", "path": AGREG_DIR / "AGGLO_PERIMETRE_SANS_LAC-parquet" / "AGGLO_PERIMETRE_SANS_LAC.parquet"},
}
PERIMETER_VARIANT = "avec_lac"

# --- Méthode ---
CH_MULTIBRUIT = True            # False = ancienne méthode (route jour seule)
EQUIV_GENE = True               # conversion fer / aérien en « équivalent gêne routière » (OMS 2018)
EQUIV_OFFSET_MIN_DB = 0.0       # L_eq >= L + 0 ; None = pas de borne

# Courbes %HA OMS 2018 (Dir. UE 2020/367, annexe III) : %HA = a + b*L + c*L^2, L = Lden
HA_COEFS = {
    "route": (78.9270, -3.1162, 0.0342),
    "fer":   (38.1596, -2.05538, 0.0285),
    "air":   (-50.9693, 1.0168, 0.0072),
}
H_JOUR, H_NUIT, PENALITE_NUIT_DB = 16, 8, 10    # périodes OPB (6h-22h / 22h-6h), pénalité Lden de nuit

FR_CLASSE_1 = "45dB"            # "45dB" | "nan"
OVERLAP_RULE = "max"            # "max" | "priorite_ch" | "priorite_fr"
RESOLUTION_M = 10               # None = résolution native la plus grossière
CH_EXTRA_NODATA = []            # ex. [0]

TARGET_CRS = "EPSG:2056"        # CH1903+ / LV95
CLASS_BREAKS_DB = [50, 55, 60, 65, 70, 75]   # seuils ORHANE (Lden), bornes basses incluses
# valeur indicative en dB par classe (milieu de classe ; 7 est ouverte vers le haut)
CLASS_TO_DB = {1: 45.0, 2: 52.5, 3: 57.5, 4: 62.5, 5: 67.5, 6: 72.5, 7: 77.5}

RESAMPLING_CH = Resampling.average   # moyenne faite en ÉNERGIE (10^(L/10)), puis retour en dB
RESAMPLING_FR = Resampling.mode      # classes : valeur majoritaire (jamais d'interpolation)

ADD_LONLAT = True
OUT_DIR = Path("sorties_bruit")
OUT_STEM = "bruit_classes_grand_geneve"

## Lecture des rasters et du périmètre

In [3]:
def describe(path):
    with rasterio.open(path) as src:
        if src.crs is None:
            raise ValueError(f"{path} n'a pas de SCR déclaré : impossible de le reprojeter.")
        print(f"{path}\n  SCR    : {src.crs.to_string()}\n  taille : {src.width} x {src.height}"
              f"\n  pixel  : {src.res[0]:g} x {src.res[1]:g}\n  nodata : {src.nodata}"
              f"\n  type   : {src.dtypes[0]} ({src.count} bande(s), seule la 1re est lue)")
        return src.crs, src.res

ch_paths = ([CH_RASTERS["route"]["jour"]] if not CH_MULTIBRUIT
            else [p for periodes in CH_RASTERS.values() for p in periodes.values()])
ch_meta = {str(p): describe(p) for p in ch_paths}
ch_crs, ch_res = ch_meta[str(CH_RASTERS["route"]["jour"])]
fr_crs, fr_res = describe(FR_TIF)
if CH_MULTIBRUIT:
    print(f"\n{CH_AIR_GPKG} : couches {pyogrio.list_layers(CH_AIR_GPKG)[:, 0].tolist()}")


def load_perimeter(variant):
    spec = PERIMETER_VARIANTS[variant]
    gdf = gpd.read_file(spec["path"]) if spec["kind"] == "shp" else gpd.read_parquet(spec["path"])
    return gdf.to_crs(TARGET_CRS).union_all()

perimeter = load_perimeter(PERIMETER_VARIANT)
print(f"\nPérimètre « {PERIMETER_VARIANT} » : {perimeter.area / 1e6:,.0f} km²")

../../Data/input/attributs/GG/bruit/laerm-strassenlaerm_tag_2056.tif
  SCR    : EPSG:2056
  taille : 38202 x 24449
  pixel  : 10 x 10
  nodata : -9999.0
  type   : float64 (1 bande(s), seule la 1re est lue)
../../Data/input/attributs/GG/bruit/laerm-strassenlaerm_nacht_2056.tif
  SCR    : EPSG:2056
  taille : 38202 x 24449
  pixel  : 10 x 10
  nodata : -9999.0
  type   : float64 (1 bande(s), seule la 1re est lue)
../../Data/input/attributs/GG/bruit/laerm-bahnlaerm_tag_2056.tif
  SCR    : PROJCS["CH1903+ / LV95",GEOGCS["CH1903+",DATUM["CH1903+",SPHEROID["Bessel 1841",6377397.155,299.1528128,AUTHORITY["EPSG","7004"]],AUTHORITY["EPSG","6150"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Hotine_Oblique_Mercator"],PARAMETER["latitude_of_center",46.9524055555556],PARAMETER["longitude_of_center",7.43958333333333],PARAMETER["azimuth",90],PARAMETER["rectified_grid_angle",90],PARAMETER["scale_factor",1],PARAMETER["false_easting",2600000],PARAMETER

## Grille commune en EPSG:2056

In [4]:
def res_in_metres(crs, res):
    if crs.is_geographic:
        return max(res) * 111_000 * np.cos(np.radians(46.2))
    return max(res) * crs.linear_units_factor[1]

res = float(RESOLUTION_M or max(res_in_metres(ch_crs, ch_res), res_in_metres(fr_crs, fr_res)))

xmin, ymin, xmax, ymax = perimeter.bounds
xmin, ymin = np.floor(xmin / res) * res, np.floor(ymin / res) * res
xmax, ymax = np.ceil(xmax / res) * res, np.ceil(ymax / res) * res
width, height = int(round((xmax - xmin) / res)), int(round((ymax - ymin) / res))
dst_transform = from_origin(xmin, ymax, res, res)

inside = ~geometry_mask([perimeter], out_shape=(height, width), transform=dst_transform)
print(f"Grille : {width} x {height} pixels de {res:g} m — {inside.sum():,} pixels dans le périmètre")

Grille : 6555 x 6893 pixels de 10 m — 22,127,419 pixels dans le périmètre


## Reprojection

On ne lit que la partie de chaque raster qui couvre le Grand Genève.

- **Côté suisse,** les dB sont moyennés **en énergie** (10^(L/10)) si plusieurs pixels source tombent dans un pixel cible. Les grilles sonBASE sont décalées d'un demi-pixel par rapport à la grille commune.
- **Côté ORHANE,** la règle `FR_CLASSE_1` est appliquée **après** la reprojection, pour que la classe majoritaire soit calculée sur les vraies classes 1 à 7.

In [5]:
def read_window_and_reproject(path, prepare, resampling, finish=None):
    with rasterio.open(path) as src:
        margin = 5 * res
        b = transform_bounds(TARGET_CRS, src.crs, xmin - margin, ymin - margin, xmax + margin, ymax + margin)
        win = from_bounds(*b, transform=src.transform).round_offsets().round_lengths()
        win = win.intersection(Window(0, 0, src.width, src.height))
        a = prepare(src.read(1, window=win).astype("float32"), src.nodata)
        out = np.full((height, width), np.nan, dtype="float32")
        reproject(a, out,
                  src_transform=src.window_transform(win), src_crs=src.crs, src_nodata=np.nan,
                  dst_transform=dst_transform, dst_crs=TARGET_CRS, dst_nodata=np.nan,
                  resampling=resampling)
        del a
    return finish(out) if finish else out


def prepare_ch(a, nodata):
    """Lr (dB) -> énergie 10^(L/10) ; nodata -> NaN."""
    bad = ~np.isfinite(a)
    if nodata is not None and not np.isnan(nodata):
        bad |= a == nodata
    for v in CH_EXTRA_NODATA:
        bad |= a == v
    a[bad] = np.nan
    np.power(np.float32(10), a / np.float32(10), out=a)
    return a


def energy_to_db(e):
    with np.errstate(divide="ignore", invalid="ignore"):
        return (10 * np.log10(e)).astype("float32")


def prepare_fr(a, nodata):
    a[~np.isin(a, np.arange(1, 8))] = np.nan          # -99 et toute valeur hors 1..7
    return a


def read_ch(path):
    return read_window_and_reproject(path, prepare_ch, RESAMPLING_CH, finish=energy_to_db)


fr_cls = read_window_and_reproject(FR_TIF, prepare_fr, RESAMPLING_FR)

# Appliqué APRÈS la reprojection : sinon la classe majoritaire serait calculée
# sans la classe 1 et gonflerait artificiellement les classes bruyantes.
if FR_CLASSE_1 == "nan":
    fr_cls[fr_cls == 1] = np.nan
elif FR_CLASSE_1 != "45dB":
    raise ValueError("FR_CLASSE_1 doit valoir '45dB' ou 'nan'")

## Multibruit suisse : Lden par source, équivalence de gêne et somme énergétique

Équations (1) à (5) de l'en-tête. Avec `CH_MULTIBRUIT = False`, `ch_db` est simplement le Lr routier de jour, comme dans l'ancienne version.

In [6]:
import shapely
from rasterio.features import rasterize


def _energy(L, offset_db=0.0):
    """10^((L+offset)/10), 0 là où L est NaN (période / source absente)."""
    e = np.power(np.float32(10), (L + np.float32(offset_db)) / np.float32(10))
    return np.nan_to_num(e, nan=0.0)


def lden_from_lr(l_jour, l_nuit):
    """Équation (1) : Lden approché à partir des Lr OPB jour (6h-22h) et nuit (22h-6h)."""
    has = np.isfinite(l_jour) | np.isfinite(l_nuit)
    e = (H_JOUR * _energy(l_jour) + H_NUIT * _energy(l_nuit, PENALITE_NUIT_DB)) / (H_JOUR + H_NUIT)
    out = energy_to_db(e)
    out[~has] = np.nan
    return out


def pct_ha(L, source):
    """Équation (3) : % de personnes fortement gênées (OMS 2018)."""
    a, b, c = HA_COEFS[source]
    return a + b * L + c * L * L


a_r, b_r, c_r = HA_COEFS["route"]
L_VERTEX_ROUTE = -b_r / (2 * c_r)                      # 45,6 dB : minimum de la courbe routière
HA_MIN_ROUTE = pct_ha(L_VERTEX_ROUTE, "route")         # 7,94 %


def road_equivalent(L, source):
    """Équation (4) : Lden routier donnant la même %HA que la source."""
    ha = pct_ha(L, source)
    disc = b_r * b_r - 4 * c_r * (a_r - ha)
    with np.errstate(invalid="ignore"):
        l_eq = (-b_r + np.sqrt(np.maximum(disc, 0))) / (2 * c_r)
    ok = np.isfinite(L) & (L >= L_VERTEX_ROUTE) & (ha >= HA_MIN_ROUTE)
    l_eq = np.where(ok, l_eq, L).astype("float32")
    if EQUIV_OFFSET_MIN_DB is not None:
        l_eq = np.where(np.isfinite(L), np.maximum(l_eq, L + EQUIV_OFFSET_MIN_DB), np.nan).astype("float32")
    return l_eq


def contours_to_raster(gdf):
    """Courbes d'égal niveau fermées -> raster du niveau de la courbe la plus haute contenant le pixel."""
    shapes, n_ouvertes = [], 0
    for geom, lvl in zip(gdf.geometry, gdf["level"]):
        for line in shapely.get_parts(geom):
            coords = np.asarray(line.coords)[:, :2]
            if len(coords) < 4:
                continue
            if not line.is_ring:
                if np.hypot(*(coords[0] - coords[-1])) > AIR_CLOSE_TOL_M:
                    n_ouvertes += 1
                    continue
                coords = np.vstack([coords, coords[:1]])
            for p in shapely.get_parts(shapely.make_valid(shapely.Polygon(coords))):
                if p.geom_type == "Polygon" and not p.is_empty:
                    shapes.append((p, float(lvl)))
    if n_ouvertes:
        print(f"    ⚠ {n_ouvertes} courbe(s) non fermée(s) ignorée(s) (écart > {AIR_CLOSE_TOL_M:g} m)")
    if not shapes:
        return np.full((height, width), np.nan, dtype="float32")
    shapes.sort(key=lambda t: t[1])                    # croissant : la plus haute courbe est brûlée en dernier
    return rasterize(shapes, out_shape=(height, width), transform=dst_transform,
                     fill=np.nan, dtype="float32")


def read_air_layer(layer):
    """Une couche du cadastre OFAC -> {aérodrome: raster Lr}, limitée au périmètre."""
    g = gpd.read_file(CH_AIR_GPKG, layer=layer, bbox=(xmin, ymin, xmax, ymax)).to_crs(TARGET_CRS)
    g = g[g.intersects(perimeter)].copy()
    g["level"] = pd.to_numeric(g[AIR_LEVEL_FIELD], errors="coerce")
    g = g[g["level"].notna()]
    out = {}
    for nom, sub in g.groupby("REG_NAME"):
        print(f"  {layer:<22} {nom:<16} {len(sub):>4} courbes, {sub['level'].min():g}–{sub['level'].max():g} dB")
        out[nom] = contours_to_raster(sub)
    return out


def energy_sum_db(rasters):
    """Somme énergétique de rasters dB (NaN = absent) ; NaN si tous absents."""
    rasters = [r for r in rasters if r is not None]
    if not rasters:
        return np.full((height, width), np.nan, dtype="float32")
    has = np.zeros((height, width), dtype=bool)
    e = np.zeros((height, width), dtype="float32")
    for r in rasters:
        has |= np.isfinite(r)
        e += _energy(r)
    out = energy_to_db(e)
    out[~has] = np.nan
    return out


if not CH_MULTIBRUIT:
    ch_db = read_ch(CH_RASTERS["route"]["jour"])          # ancienne méthode : Lr route jour
    lden_ch = {}
else:
    lden_ch = {}
    # --- route et fer (sonBASE) ---
    for source, periodes in CH_RASTERS.items():
        l_j, l_n = read_ch(periodes["jour"]), read_ch(periodes["nuit"])
        lden_ch[source] = lden_from_lr(l_j, l_n)
        del l_j, l_n
        print(f"{source:<5} : {np.isfinite(lden_ch[source]).sum():,} pixels avec un Lden")

    # --- aérien (OFAC) : jour par aérodrome (1re couche disponible), nuit = 3 heures ramenées à 8 h ---
    print("Aérien :")
    jour_par_aero = {}
    for layer in AIR_DAY_LAYERS:
        for nom, r in read_air_layer(layer).items():
            jour_par_aero.setdefault(nom, r)               # OverallTrafficDay_Lr prioritaire sur LightAircraft_Lr
    l_air_j = energy_sum_db(list(jour_par_aero.values()))
    heures = []
    for layer in AIR_NIGHT_LAYERS:
        heures.append(energy_sum_db(list(read_air_layer(layer).values())))
    has_n = np.zeros((height, width), dtype=bool)
    e_n = np.zeros((height, width), dtype="float32")
    for h in heures:
        has_n |= np.isfinite(h)
        e_n += _energy(h)
    l_air_n = energy_to_db(e_n / H_NUIT)                   # équation (2)
    l_air_n[~has_n] = np.nan
    del heures, e_n, has_n
    lden_ch["air"] = lden_from_lr(l_air_j, l_air_n)
    del l_air_j, l_air_n
    print(f"air   : {np.isfinite(lden_ch['air']).sum():,} pixels avec un Lden")

    # --- équivalence de gêne (équation 4) et somme énergétique (équation 5) ---
    if EQUIV_GENE:
        lden_ch["fer_eq"] = road_equivalent(lden_ch["fer"], "fer")
        lden_ch["air_eq"] = road_equivalent(lden_ch["air"], "air")
    else:
        lden_ch["fer_eq"], lden_ch["air_eq"] = lden_ch["fer"], lden_ch["air"]
    ch_db = energy_sum_db([lden_ch["route"], lden_ch["fer_eq"], lden_ch["air_eq"]])
    lden_ch["multi"] = ch_db

    # Contrôle : écart moyen de l'équivalence, par tranche de Lden
    for s in ("fer", "air"):
        m = inside & np.isfinite(lden_ch[s])
        if m.any():
            d = pd.DataFrame({"lden": lden_ch[s][m], "ecart": (lden_ch[f"{s}_eq"] - lden_ch[s])[m]})
            d["tranche"] = pd.cut(d["lden"], [0, 45, 50, 55, 60, 65, 70, 75, 200], right=False)
            print(f"\nÉcart {s}_eq − {s} (dB) par tranche de Lden {s} :")
            print(d.groupby("tranche", observed=True)["ecart"].agg(["count", "mean", "max"]).round(2).to_string())

route : 5,716,618 pixels avec un Lden
fer   : 1,051,714 pixels avec un Lden
Aérien :
  OverallTrafficDay_Lr   CDB Genève         18 courbes, 55–70 dB
  LightAircraft_Lr       CDB Genève         22 courbes, 55–70 dB
  LightAircraft_Lr       CDB La Côte         4 courbes, 50–65 dB
  FirstNightHour_Lr      CDB Genève         27 courbes, 45–65 dB
    ⚠ 1 courbe(s) non fermée(s) ignorée(s) (écart > 50 m)
  SecondNightHour_Lr     CDB Genève         23 courbes, 45–60 dB
  LastNightHour_Lr       CDB Genève          7 courbes, 45–50 dB
air   : 1,248,739 pixels avec un Lden

Écart fer_eq − fer (dB) par tranche de Lden fer :
            count  mean   max
tranche                      
[0, 45)    683492  0.00  0.00
[45, 50)    41687  0.00  0.00
[50, 55)    27824  0.03  0.51
[55, 60)    17981  1.45  2.22
[60, 65)    12391  2.58  2.87
[65, 70)     6840  2.94  3.12
[70, 75)       33  3.16  3.17
[75, 200)      20  3.15  3.17

Écart air_eq − air (dB) par tranche de Lden air :
           count   mean    

## Reclassement de la couche suisse

`np.digitize` avec les seuils `[50, 55, 60, 65, 70, 75]` donne 0 pour < 50,
1 pour 50–55, …, 6 pour ≥ 75 ; on ajoute 1 pour obtenir les classes 1 à 7
(bornes basses incluses, comme dans le tableau ORHANE : 50 ≤ Lden < 55 → 2).

In [7]:
ch_cls = np.where(np.isfinite(ch_db), np.digitize(ch_db, CLASS_BREAKS_DB) + 1, np.nan).astype("float32")

for name, arr in [("Suisse", ch_cls), ("ORHANE", fr_cls)]:
    v = arr[inside & np.isfinite(arr)].astype(int)
    counts = pd.Series(v).value_counts().sort_index()
    print(f"{name} — pixels par classe dans le périmètre :\n{counts.to_string()}\n")

Suisse — pixels par classe dans le périmètre :
1    2991969
2     501413
3     423035
4     252797
5     330187
6     302840
7     162810

ORHANE — pixels par classe dans le périmètre :
1    11274396
2     1177021
3      805632
4      618574
5      330011
6      147121
7       92592



## Union, découpe et normalisation 0–1

In [8]:
has_ch, has_fr = np.isfinite(ch_cls), np.isfinite(fr_cls)
both = has_ch & has_fr

classe = np.where(has_ch, ch_cls, fr_cls)
if OVERLAP_RULE == "max":
    classe[both] = np.maximum(ch_cls[both], fr_cls[both])
elif OVERLAP_RULE == "priorite_ch":
    classe[both] = ch_cls[both]
elif OVERLAP_RULE == "priorite_fr":
    classe[both] = fr_cls[both]
else:
    raise ValueError(f"OVERLAP_RULE inconnue : {OVERLAP_RULE}")
classe[~inside] = np.nan

classe_norm = (classe - 1) / 6                       # 1 -> 0, 7 -> 1

# Valeur indicative en dB : Lr suisse réel côté CH, milieu de classe côté FR
lut = np.full(8, np.nan, dtype="float32")
for k, v in CLASS_TO_DB.items():
    lut[k] = v
fr_db = np.where(has_fr, lut[np.nan_to_num(fr_cls, nan=0).astype(int)], np.nan)
choix_ch = has_ch & (~both | (classe == ch_cls))
bruit_db = np.where(choix_ch, ch_db, fr_db)

source = np.select([both, has_ch, has_fr], ["CH+FR", "CH", "FR"], default="")

print(f"Pixels avec une classe : {np.isfinite(classe).sum():,} "
      f"(dont {(both & inside).sum():,} en recouvrement, règle « {OVERLAP_RULE} »)")
print(f"Pixels du périmètre sans donnée (lac, hors couverture, classe 1 FR si 'nan') : "
      f"{(inside & ~np.isfinite(classe)).sum():,}")

Pixels avec une classe : 19,152,901 (dont 257,497 en recouvrement, règle « max »)
Pixels du périmètre sans donnée (lac, hors couverture, classe 1 FR si 'nan') : 2,974,518


## Export

Colonnes : `classe` (1–7), `classe_norm` (0–1), `bruit_db_approx` (Lr suisse
réel ou valeur indicative ORHANE), `classe_ch`, `classe_fr` (chaque source
seule, vide si absente), `source` (`CH`, `FR`, `CH+FR`), coordonnées LV95
(+ `lon`, `lat`).

Le GeoTIFF contient les classes en entiers (0 = pas de donnée) avec la palette
ORHANE intégrée : QGIS l'affiche directement avec les bonnes couleurs.

In [9]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

PALETTE = {1: "#6ee86e", 2: "#c9ff8f", 3: "#ffff3c", 4: "#f5c04a",
           5: "#ec3a2a", 6: "#c63cf5", 7: "#6f2f9f"}
hex2rgba = lambda h: tuple(int(h[i:i + 2], 16) for i in (1, 3, 5)) + (255,)

with rasterio.open(OUT_DIR / f"{OUT_STEM}.tif", "w", driver="GTiff", width=width, height=height,
                   count=1, dtype="uint8", crs=TARGET_CRS, transform=dst_transform,
                   nodata=0, compress="deflate") as dst:
    dst.write(np.nan_to_num(classe, nan=0).astype("uint8"), 1)
    dst.write_colormap(1, {0: (0, 0, 0, 0), **{k: hex2rgba(v) for k, v in PALETTE.items()}})

# Colonnes de contrôle côté CH (multibruit) : une bande par niveau dans un GeoTIFF à part
CONTROL_KEYS = ["route", "fer", "air", "fer_eq", "air_eq", "multi"]
if CH_MULTIBRUIT:
    with rasterio.open(OUT_DIR / f"{OUT_STEM}_lden_ch.tif", "w", driver="GTiff", width=width, height=height,
                       count=len(CONTROL_KEYS), dtype="float32", crs=TARGET_CRS, transform=dst_transform,
                       nodata=np.nan, compress="deflate", predictor=3, tiled=True) as dst:
        for i, k in enumerate(CONTROL_KEYS, start=1):
            band = np.where(inside, lden_ch[k], np.nan).astype("float32")
            dst.write(band, i)
            dst.set_band_description(i, f"lden_{k}" if k != "multi" else "lden_multi_ch")

rows, cols = np.nonzero(np.isfinite(classe))
xs = xmin + (cols + 0.5) * res
ys = ymax - (rows + 0.5) * res

df = pd.DataFrame({
    "x_lv95": xs,
    "y_lv95": ys,
    "classe": classe[rows, cols].astype("int8"),
    "classe_norm": classe_norm[rows, cols].round(4),
    "bruit_db_approx": bruit_db[rows, cols].round(1),
    "classe_ch": pd.array(ch_cls[rows, cols], dtype="Float32").astype("Int8"),
    "classe_fr": pd.array(fr_cls[rows, cols], dtype="Float32").astype("Int8"),
    "source": source[rows, cols],
})
if CH_MULTIBRUIT:
    for k in CONTROL_KEYS:
        col = f"lden_{k}" if k != "multi" else "lden_multi_ch"
        df[col] = np.round(lden_ch[k][rows, cols], 1).astype("float32")

gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x_lv95, df.y_lv95), crs=TARGET_CRS)
if ADD_LONLAT:
    wgs = gdf.geometry.to_crs(4326)
    gdf["lon"], gdf["lat"] = wgs.x.round(6).to_numpy(), wgs.y.round(6).to_numpy()

gdf.to_file(OUT_DIR / f"{OUT_STEM}.gpkg", layer="bruit_classes", driver="GPKG", engine="pyogrio")
gdf.to_parquet(OUT_DIR / f"{OUT_STEM}.parquet")
gdf.drop(columns="geometry").to_csv(OUT_DIR / f"{OUT_STEM}.csv", index=False)

for f in sorted(OUT_DIR.glob(f"{OUT_STEM}*.*")):
    print(f"{f}  ({f.stat().st_size / 1e6:,.1f} Mo)")
gdf.head()

sorties_bruit/bruit_classes_grand_geneve.csv  (1,249.8 Mo)
sorties_bruit/bruit_classes_grand_geneve.gpkg  (2,874.9 Mo)
sorties_bruit/bruit_classes_grand_geneve.parquet  (344.6 Mo)
sorties_bruit/bruit_classes_grand_geneve.tif  (1.5 Mo)
sorties_bruit/bruit_classes_grand_geneve_lden_ch.tif  (29.8 Mo)


,x_lv95,y_lv95,classe,classe_norm,bruit_db_approx,classe_ch,classe_fr,source,lden_route,lden_fer,lden_air,lden_fer_eq,lden_air_eq,lden_multi_ch,geometry,lon,lat
0,2501665.0,1155685.0,1,0.0,43.900002,1,<NA>,CH,43.900002,NaN,NaN,NaN,NaN,43.900002,POINT (2501665 1155685),6.156534,46.545208
1,2501675.0,1155685.0,1,0.0,43.700001,1,<NA>,CH,43.700001,NaN,NaN,NaN,NaN,43.700001,POINT (2501675 1155685),6.156664,46.545209
2,2501655.0,1155675.0,1,0.0,43.700001,1,<NA>,CH,43.700001,NaN,NaN,NaN,NaN,43.700001,POINT (2501655 1155675),6.156405,46.545116
3,2501665.0,1155675.0,1,0.0,43.400002,1,<NA>,CH,43.400002,NaN,NaN,NaN,NaN,43.400002,POINT (2501665 1155675),6.156536,46.545118
4,2501675.0,1155675.0,1,0.0,43.200001,1,<NA>,CH,43.200001,NaN,NaN,NaN,NaN,43.200001,POINT (2501675 1155675),6.156666,46.545119


## Contrôle rapide

In [10]:
print(pd.crosstab(gdf["classe"], gdf["source"], margins=True))
print()
print(gdf.groupby("classe")["classe_norm"].first())

if CH_MULTIBRUIT:
    ch = gdf[gdf["source"].isin(["CH", "CH+FR"])]
    print("\nCôté CH — pixels où chaque source est présente, et Lden moyen :")
    print(pd.DataFrame({
        "pixels": ch[["lden_route", "lden_fer", "lden_air"]].notna().sum(),
        "lden_moyen": ch[["lden_route", "lden_fer", "lden_air"]].mean().round(1),
    }).to_string())
    # Ce que le multibruit change par rapport à la route seule
    cls_route = pd.Series(np.digitize(ch["lden_route"].fillna(0), CLASS_BREAKS_DB) + 1, index=ch.index)
    print("\nClasse « route seule » (lignes) vs classe multibruit CH (colonnes) :")
    print(pd.crosstab(cls_route.rename("route_seule"), ch["classe_ch"].rename("multibruit")))

source       CH   CH+FR        FR       All
classe                                     
1       2778467  104748  11166942  14050157
2        498410   23842   1153442   1675694
3        422340   17145    788393   1227878
4        252453   53470    563129    869052
5        316047   24471    303035    643553
6        288641   17366    135740    441747
7        151196   16455     77169    244820
All     4707554  257497  14187850  19152901

classe
1    0.0000
2    0.1667
3    0.3333
4    0.5000
5    0.6667
6    0.8333
7    1.0000
Name: classe_norm, dtype: float32

Côté CH — pixels où chaque source est présente, et Lden moyen :
             pixels  lden_moyen
lden_route  4416633   43.400002
lden_fer     793604   30.700001
lden_air    1248739   51.000000

Classe « route seule » (lignes) vs classe multibruit CH (colonnes) :
multibruit         1       2       3       4       5       6      7
route_seule                                                        
1            2987406   22885   6561

In [11]:
# === Export agrégé au carroyage 200 m (SITG, AGGLO_CARREAU_200) ===
# Chaque pixel de la grille commune est rattaché au carreau qui contient son centre
# (à 10 m : exactement 20 x 20 = 400 pixels par carreau, aucun pixel n'est coupé).
# Source carroyage : « Système d'information du territoire à Genève (SITG), extrait en
# date du 31.03.2025 » — agrégation du bruit réalisée sur cette base (CGU SITG, art. 9).

CARREAU_PATH = AGREG_DIR / "AGGLO_CARREAU_200-SHP" / "AGGLO_CARREAU_200.shp"
CARREAU_ID = "GRID_ID"
CARREAU_MIN_COUVERTURE = 0.0     # part mini de pixels avec donnée pour garder un carreau (0 = au moins 1 pixel)
CARREAU_STEM = f"{OUT_STEM}_carreau200"

carreaux = gpd.read_file(CARREAU_PATH).to_crs(TARGET_CRS).reset_index(drop=True)

# --- géométrie de la grille, déduite des carreaux eux-mêmes ---
bds = carreaux.geometry.bounds
pas = float(np.median(bds.maxx - bds.minx))
x0, y0 = float(bds.minx.min()), float(bds.miny.min())
ci = np.round((bds.minx - x0) / pas).astype(int).to_numpy()
cj = np.round((bds.miny - y0) / pas).astype(int).to_numpy()
assert np.allclose(bds.minx - x0, ci * pas, atol=1) and np.allclose(bds.miny - y0, cj * pas, atol=1), \
    "Carroyage non régulier : l'indexation arithmétique ne s'applique pas."
ncol, nrow = ci.max() + 1, cj.max() + 1
cell_lut = np.full(nrow * ncol, -1, dtype=np.int64)
cell_lut[cj * ncol + ci] = np.arange(len(carreaux))

if abs(pas / res - round(pas / res)) > 1e-6:
    print(f"⚠️ {pas:g} m n'est pas un multiple de la résolution ({res:g} m) : "
          "le nombre de pixels par carreau varie, la couverture est approximative.")
n_theo = (pas / res) ** 2

# --- rattachement pixel -> carreau (pixels du périmètre seulement) ---
r, c = np.nonzero(inside)
px = xmin + (c + 0.5) * res
py = ymax - (r + 0.5) * res
pi = np.floor((px - x0) / pas).astype(np.int64)
pj = np.floor((py - y0) / pas).astype(np.int64)
dans_grille = (pi >= 0) & (pi < ncol) & (pj >= 0) & (pj < nrow)
idx = np.full(r.shape, -1, dtype=np.int64)
idx[dans_grille] = cell_lut[pj[dans_grille] * ncol + pi[dans_grille]]
garde = idx >= 0
r, c, idx = r[garde], c[garde], idx[garde]
print(f"{pas:g} m, {len(carreaux):,} carreaux — {garde.sum():,} pixels du périmètre rattachés "
      f"({(~garde).sum():,} hors carroyage)")

n = len(carreaux)
cnt = lambda m, w=None: np.bincount(idx[m], weights=None if w is None else w[m], minlength=n)

cls_px = classe[r, c]
ok = np.isfinite(cls_px)
n_perim = np.bincount(idx, minlength=n)
n_data = cnt(ok)

# effectifs par classe -> parts, classe majoritaire, classe max
par_classe = np.stack([cnt(ok & (cls_px == k)) for k in range(1, 8)], axis=1)   # (n, 7)
with np.errstate(invalid="ignore", divide="ignore"):
    parts = par_classe / n_data[:, None]
    classe_norm_moy = cnt(ok, classe_norm[r, c]) / n_data
    classe_moy = cnt(ok, cls_px) / n_data
    # moyenne énergétique en dB : 10·log10(moyenne de 10^(L/10)) — moyenne physique d'un niveau sonore
    db_px = bruit_db[r, c]
    okdb = np.isfinite(db_px)
    db_energ = 10 * np.log10(cnt(okdb, 10 ** (db_px / 10)) / cnt(okdb))
    part_fr = cnt(ok & ~np.isfinite(ch_cls[r, c])) / n_data

has = n_data > 0
classe_mode = np.where(has, par_classe.argmax(axis=1) + 1, 0)
classe_max = np.where(has, 7 - np.argmax(par_classe[:, ::-1] > 0, axis=1), 0)

stats = pd.DataFrame({
    "n_pix_perimetre": n_perim,
    "n_pix_donnee": n_data,
    "couverture": (n_data / n_theo).round(4),               # part du carreau avec une classe de bruit
    "classe_norm_moy": classe_norm_moy.round(4),            # 0 (calme) – 1 (classe 7)
    "classe_moy": classe_moy.round(3),
    "classe_mode": pd.array(classe_mode, dtype="Int8"),
    "classe_max": pd.array(classe_max, dtype="Int8"),
    "part_classe_ge4": parts[:, 3:].sum(axis=1).round(4),   # part ≥ 60 dB (zone « altérée » et pire)
    **{f"part_classe_{k}": parts[:, k - 1].round(4) for k in range(1, 8)},
    "bruit_db_energ": np.round(db_energ, 1),
    "part_pix_fr": part_fr.round(4),                        # part des pixels venant d'ORHANE
})
stats.loc[~has, ["classe_mode", "classe_max"]] = pd.NA

gdf_c = gpd.GeoDataFrame(pd.concat([carreaux.drop(columns="geometry"), stats], axis=1),
                         geometry=carreaux.geometry, crs=TARGET_CRS)
gdf_c = gdf_c[has & (stats["couverture"] >= CARREAU_MIN_COUVERTURE)].reset_index(drop=True)
centres = gdf_c.geometry.centroid
gdf_c["x_lv95"], gdf_c["y_lv95"] = centres.x.round(1), centres.y.round(1)

gdf_c.to_file(OUT_DIR / f"{CARREAU_STEM}.gpkg", layer="bruit_carreau200", driver="GPKG", engine="pyogrio")
gdf_c.to_parquet(OUT_DIR / f"{CARREAU_STEM}.parquet")
gdf_c.drop(columns="geometry").to_csv(OUT_DIR / f"{CARREAU_STEM}.csv", index=False)

print(f"{len(gdf_c):,} carreaux exportés "
      f"(dont {(gdf_c['couverture'] < 0.999).sum():,} partiellement couverts : bord, lac, frontière de données)")
for f in sorted(OUT_DIR.glob(f"{CARREAU_STEM}.*")):
    print(f"{f}  ({f.stat().st_size / 1e6:,.1f} Mo)")
gdf_c[[CARREAU_ID, "couverture", "classe_norm_moy", "classe_mode", "classe_max",
       "part_classe_ge4", "bruit_db_energ"]].describe(include="all")

200 m, 166,762 carreaux — 22,127,419 pixels du périmètre rattachés (0 hors carroyage)
49,731 carreaux exportés (dont 4,222 partiellement couverts : bord, lac, frontière de données)
sorties_bruit/bruit_classes_grand_geneve_carreau200.csv  (6.9 Mo)
sorties_bruit/bruit_classes_grand_geneve_carreau200.gpkg  (15.1 Mo)
sorties_bruit/bruit_classes_grand_geneve_carreau200.parquet  (2.6 Mo)


,GRID_ID,couverture,classe_norm_moy,classe_mode,classe_max,part_classe_ge4,bruit_db_energ
count,49731,49731.000000,49731.000000,49731.0,49731.0,49731.000000,49731.000000
unique,49731,NaN,NaN,<NA>,<NA>,NaN,NaN
top,JU-367,NaN,NaN,<NA>,<NA>,NaN,NaN
freq,1,NaN,NaN,<NA>,<NA>,NaN,NaN
mean,NaN,0.962825,0.110122,1.563954,2.280791,0.111734,48.696988
std,NaN,0.158676,0.209394,1.237845,1.925973,0.268323,10.070065
min,NaN,0.002500,0.000000,1.0,1.0,0.000000,-1.600000
25%,NaN,1.000000,0.000000,1.0,1.0,0.000000,45.000000
50%,NaN,1.000000,0.000000,1.0,1.0,0.000000,45.000000
75%,NaN,1.000000,0.134200,1.0,3.0,0.000000,53.000000


## Sources à citer

- **Périmètre :** « Source : Système d'information du territoire à Genève (SITG), extrait en date du 31.03.2025 », avec mention du traitement (CGU SITG, art. 9).
- **ORHANE :** « Source ATMO Auvergne-Rhône-Alpes / ACOUCITE / CEREMA », licence ODbL (les jeux de données dérivés sont publiés sous ODbL).
- **Bruit route et fer, Suisse :** « Source : Office fédéral de l'environnement (OFEV), sonBASE », couches `ch.bafu.laerm-strassenlaerm_*` et `ch.bafu.laerm-bahnlaerm_*` (conditions d'utilisation opendata.swiss, mention de la source obligatoire).
- **Bruit aérien, Suisse :** « Source : Office fédéral de l'aviation civile (OFAC), cadastres de bruit des aérodromes civils », `ch.bazl.laermbelastungskataster-zivilflugplaetze`.
- **Méthode :**
  - Lden approché à partir des Lr OPB jour et nuit ;
  - conversion du fer et de l'aérien en « équivalent gêne routière » avec les courbes %HA de l'OMS 2018 (Directive (UE) 2020/367, annexe III), comme ORHANE ;
  - somme énergétique, puis classes ORHANE.

  Équations et références complètes dans l'en-tête du notebook.